# Demo notebook

The purpose of this notebook is to demonstrate the functionality of `seagliderOG1` to convert from Seaglider basestation files to OG1 format.

- OG1 format is a newly agreed format (since June 2024) for glider data sets from various platforms (e.g., Seaglider, Slocum, Seaexplorer).  It lives on github here: (https://github.com/OceanGlidersCommunity/OG-format-user-manual).
- OG1 manual: https://oceangliderscommunity.github.io/OG-format-user-manual/OG_Format.html

The bundled sample is sg271 from the Drygalski Trough (PycnMix), March 2026, shipped inside the package (`seagliderOG1/data/sg271/`).

The demo is organised to show

- Conversion of a single dive cycle (single `p*.nc` file)

- Conversion for a folder of local dive-cycle files (full mission of `p*.nc` files)

- Download from remote server + conversion (directory with full mission of `p*.nc` files)

Options are provided to only load e.g. 10 files, but note that OG1 format expects a full mission.

In [1]:
import pathlib
import warnings

import xarray as xr

import seagliderOG1 as sg
from seagliderOG1 import plotters, readers, writers

warnings.simplefilter(action="ignore", category=FutureWarning)

# Repo root, used to locate the bundled sample data under data/.
parent_dir = pathlib.Path().absolute().parents[0]
print(parent_dir)

/home/runner/work/seagliderOG1/seagliderOG1


In [2]:
# Specify the path for writing datafiles
data_path = parent_dir / "data"

## Reading basestation files

This has three ways to load a glider dataset.

Load an example dataset using `seagliderOG1.readers.load_sample_basestation`

Alternatively, use your own with e.g. `ds = xr.open_dataset('/path/to/yourfile.nc')`

### Load single sample dataset

In [3]:
ds = readers.load_sample_basestation()
ds

<xarray.Dataset> Size: 218kB
Dimensions:                                       (gc_event: 3, gc_state: 5,
                                                   gps_info: 3,
                                                   legato_data_point: 492,
                                                   tridentebb700chla470fdom365_data_point: 246,
                                                   depth_data_point: 89,
                                                   codaTODO_data_point: 246,
                                                   sg_data_point: 89,
                                                   ctd_data_point: 492,
                                                   magnetometer_data_point: 89,
                                                   trajectory: 1)
Coordinates:
    ctd_time                                      (ctd_data_point) datetime64[ns] 4kB ...
    ctd_depth                                     (ctd_data_point) float64 4kB ...
    latitude                                      (ctd_data_point) float64 4kB ...
    longitude                                     (ctd_data_point) float64 4kB ...
  * trajectory                                    (trajectory) int32 4B 1
Dimensions without coordinates: gc_event, gc_state, gps_info,
                                legato_data_point,
                                tridentebb700chla470fdom365_data_point,
                                depth_data_point, codaTODO_data_point,
                                sg_data_point, ctd_data_point,
                                magnetometer_data_point
Data variables: (12/509)
    sg_cal_id_str                                 |S3 3B ...
    sg_cal_mission_title                          |S17 17B ...
    sg_cal_mass                                   float64 8B ...
    sg_cal_sg_ct_type                             int32 4B ...
    sg_cal_calibcomm                              |S42 42B ...
    sg_cal_legato_sealevel                        float64 8B ...
    ...                                            ...
    eng_mag_z                                     (magnetometer_data_point) float64 712B ...
    glider                                        |S15 15B ...
    codaTODO                                      |S8 8B ...
    tridentebb700chla470fdom365                   |S27 27B ...
    legato                                        |S6 6B ...
    magnetometer                                  |S12 12B ...
Attributes: (12/48)
    time_coverage_start:             2026-03-02T19:03:07Z
    time_coverage_end:               2026-03-02T19:03:53Z
    geospatial_lat_min:              -72.97984082491809
    geospatial_lat_max:              -72.97946730813256
    geospatial_lon_min:              171.40895
    geospatial_lon_max:              171.41208246924162
    ...                              ...
    Metadata_Conventions:            Unidata Dataset Discovery v1.0
    Conventions:                     CF-1.6
    standard_name_vocabulary:        CF-1.6
    featureType:                     trajectory
    cdm_data_type:                   Trajectory
    nodc_template_version:           NODC_NetCDF_Trajectory_Template_v0.9

### Load datasets from a local directory

In [4]:
# Point at a directory of basestation files. Here, the bundled sg271 sample inside the package.
input_dir = readers.sample_mission_dir()

# Load and concatenate all datasets in the input directory
# Optionally, specify the range of profiles to load (start_profile, end_profile)
list_datasets = readers.read_mission(input_dir, start_profile=1, end_profile=3)

# Where list_datasets is a list of xarray datasets.  A single dataset can be accessed as
ds = list_datasets[0]

Loading datasets:   0%|          | 0/3 [00:00<?, ?file/s]

Loading datasets:  67%|██████▋   | 2/3 [00:00<00:00, 11.51file/s]

Loading datasets: 100%|██████████| 3/3 [00:00<00:00, 11.10file/s]

In [5]:
ds = readers.load_sample_basestation()

### Load datasets from a remote directory (URL)

In [6]:
# Specify the server where data are located
# server = "https://www.ncei.noaa.gov/data/oceans/glider/seaglider/uw/033/20100903/"

# Load and concatenate all datasets from the server, optionally specifying the range of profiles to load
# list_datasets = readers.read_mission(server, start_profile=1, end_profile=10)

## Convert to OG1 format

Process:

1. For one basestation dataset, split the dataset by dimension (`split_ds`)
3. Transform into OG1 format: dataset with dims `sg_data_point`
    - Change the dimension to `N_MEASUREMENTS`
    - Rename variables according to `vocabularies.standard_names` 
    - Assign variable attributes according to `vocabularies.vocab_attrs`.  (Note: This *could* go wrong since it makes assumptions about the input variables. May need additional handling.)
4. Add missing mandatory variables: 
    - From `split_ds[(gps_info,)]`, add the `LATITUDE_GPS`, `LONGITUDE_GPS` and `TIME_GPS` (Note: presently `TIME_GPS` is stripped before saving, but `TIME` values contain `TIME_GPS`)
    - Create `PROFILE_NUMBER` and `PHASE`
    - Calculate `DEPTH_Z` which is positive up
5. Update attributes for the file. 
    - Combines `creator` and `contributor` from original attributes into `contributor`
    - Adds `contributing_institutions` based on `institution`
    - Reformats time in `time_coverage_*` and `start_time`--> `start_date`
    - Adds `date_modified`
    - Renames `comments`-->`history`, `site`-->`summary`
    - Adds `title`, `platform`, `platform_vocabulary`, `featureType`, `Conventions`, `rtqc_method*` according to OceanGliders format
    - Retains `naming_authority`, `institution`, `project`, `geospatial_*` as OG attributes
    - Retains extra attributes: `license`, `keywords`, `keywords_vocabulary`, `file_version`, `acknowledgement`, `disclaimer`

Future behaviour to be added:

6. Retain the variables starting with `sg_cal` and check whether they vary over the mission (shouldn't)
6. Add sensors, using information in the `split_ds` with no dimensions
    - Need (from sg_cal_constants: `sg_cal` plus `volmax`, `vbd_cnts_per_cc`, `therm_expan`, `t_*`, `mass`, `hd_*`, `ctcor`, `cpcor`, `c_*`, `abs_compress`, `a`, `Tcor`, `Soc`, `Pcor`, `Foffset`)
    - Maybe also `reviewed`, `magnetic_variation` (which will change with position), `log_D_FLARE`, `flight_avg_speed_north` and `flight_avg_speed_east` also with `_gsm`, `depth_avg_curr_north` and `depth_avg_curr_east` also with `_gsm`, 
    `wlbb2f` - means sensor
    `sg_cal_mission_title`
    `sg_cal_id_str`
    `calibcomm_oxygen`
    `calibcomm`
    `sbe41` means ??
    `hdm_qc`
    `glider`
    
### Convert a single (sample) dataset

In [7]:
# Loads the bundled sg271 sample (first dive) via load_sample_basestation()
ds = readers.load_sample_basestation()

ds_OG1 = sg.convert(
    [ds],
    contributors=[{"name": "Jane Doe", "role": "PI", "email": "jane@example.org"}],
    institutions=[{"name": "University of Hamburg", "role": "Operator"}],
)

# Check the results - uncomment the following lines to either generate a plot or show the variables.
plotters.plot_profile_depth(ds_OG1)

Processing datasets:   0%|          | 0/1 [00:00<?, ?dataset/s]

speed_gsm: cm/s -> m s-1 x0.01


horz_speed_gsm: cm/s -> m s-1 x0.01


vert_speed_gsm: cm/s -> m s-1 x0.01


CNDC_RAW: S/m -> mS cm-1 x10


CNDC: S/m -> mS cm-1 x10


... (14 outputs left out)

sg_cal_max_stall_speed: cm/s -> m s-1 x0.01


surface_curr_east: cm/s -> m s-1 x0.01


surface_curr_north: cm/s -> m s-1 x0.01


(<Figure size 1000x600 with 1 Axes>,
 <Axes: title={'center': 'Profile Depth as a Function of Time'}, xlabel='Time (2026)', ylabel='Depth'>)

In [8]:
### list the variables of the converted dataset
sorted(ds_OG1.variables)

['BUOYANCY',
 'CDOM',
 'CHLA',
 'CNDC',
 'CNDC_QC',
 'CNDC_RAW',
 'CNDC_RAW_QC',
 'CTD_qc',
 'C_VBD',
 'DAVG_CURR_EAST',
 'DAVG_CURR_EAST_QC',
 'DAVG_CURR_NORTH',
 'DELTA_TIME_S',
 'DENSITY',
 'DEPLOYMENT_LATITUDE',
 'DEPLOYMENT_LONGITUDE',
 'DEPLOYMENT_TIME',
 'DEPTH',
 'DEPTH_2',
 'DEPTH_3',
 'DEPTH_Z',
 'EAST_DISPLACEMENT',
 'GLIDER_FIRMWARE_VERSION',
 'GLIDER_HORZ_VELO_MODEL',
 'GLIDER_VERT_VELO_MODEL',
 'GLIDE_ANGLE',
 'GLIDE_SPEED',
 'GLIDE_SPEED_QC',
 'GPS1_qc',
 'GPS2_qc',
 'GPSE_qc',
 'GPS_east_displacement_m',
 'GPS_north_displacement_m',
 'HD_A',
 'HD_B',
 'HD_C',
 'HEADING',
 'LANDSTATION_VERSION',
 'LATITUDE',
 'LATITUDE_GPS',
 'LONGITUDE',
 'LONGITUDE_GPS',
 'MASS',
 'NORTH_DISPLACEMENT',
 'OXYSAT',
 'PHASE',
 'PHASE_QC',
 'PITCH',
 'PLATFORM_MAKER',
 'PLATFORM_MODEL',
 'PLATFORM_SERIAL_NUMBER',
 'POTDENS0',
 'PRES',
 'PRES_2',
 'PRES_3',
 'PRES_QC',
 'PROFILE_DIRECTION',
 'PROFILE_NUMBER',
 'PSAL',
 'PSAL_QC',
 'PSAL_RAW',
 'PSAL_RAW_QC',
 'ROLL',
 'SEGMENT_NUMBER',
 'SE

In [9]:
# Print to screen a table of attributes
plotters.show_contents(ds_OG1, "attrs")

,Attribute,Value,DType
0,conversion_units,"{""speed_gsm"": [""cm/s"", ""m s-1"", 0.01], ""horz_s...",str
1,conversion_sensor_calibration,"{""SENSOR_CTD_238007"": {""sensor_key"": ""legato"",...",str
2,title,OceanGliders trajectory file,str
3,id,sg271_20260302T193828_R,str
4,platform,sub-surface gliders,str
5,platform_vocabulary,https://vocab.nerc.ac.uk/collection/L06/curren...,str
6,geospatial_lat_min,-72.98065,ndarray
7,geospatial_lat_max,-72.97946730813256,ndarray
8,geospatial_lon_min,171.4084666666667,ndarray
9,geospatial_lon_max,171.41705000000002,ndarray


In [10]:
# Print to screen a table of the variables and variable attributes
plotters.show_contents(ds_OG1, "variables")

,dims,units,comment,standard_name,dtype
name,,,,,
BUOYANCY,N_MEASUREMENTS,g,"Buoyancy of vehicle, corrected for compression effects",,float32
CDOM,N_MEASUREMENTS,1e-9,fDOM fluorescence (365nm excitation/450m emission),,float32
CHLA,N_MEASUREMENTS,mg m-3,chlorophyll-a concentration (470nm excitation/695nm emission) scaled to the fluorescence response from a monoculture of Thalassiosira weissflogii.,mass_concentration_of_chlorophyll_a_in_sea_water,float32
CNDC,N_MEASUREMENTS,mS cm-1,Conductivity corrected for anomalies converted from S/m (x10),sea_water_electrical_conductivity,float32
CNDC_QC,N_MEASUREMENTS,,Whether to trust each corrected conductivity value,status_flag,int8
CNDC_RAW,N_MEASUREMENTS,mS cm-1,Uncorrected conductivity converted from S/m (x10),sea_water_electrical_conductivity,float32
CNDC_RAW_QC,N_MEASUREMENTS,,Whether to trust each raw conductivity value,status_flag,int8
DELTA_TIME_S,N_MEASUREMENTS,seconds,Difference between sample times,,float64
DENSITY,N_MEASUREMENTS,kg m-3,"raw, uncorrected salinity units label 'g/m^3' in the basestation file corrected to 'kg m-3'",sea_water_density,float32


### Convert mission from a local directory of basestation files

- For local data in the directory `input_dir`
- Creates a plot of ctd_depth against ctd_time.


In [11]:
# Point at a directory of basestation files (the bundled sg271 sample).
input_dir = readers.sample_mission_dir()

# Load and concatenate all datasets in the input directory
# Optionally, specify the range of profiles to load (start_profile, end_profile)
list_datasets = readers.read_mission(input_dir, start_profile=1, end_profile=3)

# Convert the list of datasets to OG1
ds_OG1 = sg.convert(
    list_datasets,
    contributors=[{"name": "Jane Doe", "role": "PI", "email": "jane@example.org"}],
    institutions=[{"name": "University of Hamburg", "role": "Operator"}],
)

# Generate a simple plot
plotters.plot_profile_depth(ds_OG1)
plotters.show_contents(ds_OG1, "attrs")

Loading datasets:   0%|          | 0/3 [00:00<?, ?file/s]

Loading datasets:  67%|██████▋   | 2/3 [00:00<00:00, 12.36file/s]

Loading datasets: 100%|██████████| 3/3 [00:00<00:00, 12.23file/s]

Processing datasets:   0%|          | 0/3 [00:00<?, ?dataset/s]

Processing datasets:  33%|███▎      | 1/3 [00:00<00:01,  1.52dataset/s]

Processing datasets:  67%|██████▋   | 2/3 [00:01<00:00,  1.04dataset/s]

Processing datasets: 100%|██████████| 3/3 [00:03<00:00,  1.31s/dataset]

Processing datasets: 100%|██████████| 3/3 [00:03<00:00,  1.19s/dataset]


Sensors ['codaTODO', 'tridentebb700chla470fdom365'] share serial number 237790; a calibration comment may have been copied from another instrument. Correct it with sensors: in the mission config.


/tmp/ipykernel_2522/3117069022.py:9: UserWarning: PLATFORM_MODEL not set in the mission config (platform.PLATFORM_MODEL); writing 'UNK'.
  ds_OG1 = sg.convert(
/tmp/ipykernel_2522/3117069022.py:9: UserWarning: PLATFORM_MAKER not set in the mission config (platform.PLATFORM_MAKER); writing 'UNK'.
  ds_OG1 = sg.convert(
/tmp/ipykernel_2522/3117069022.py:9: UserWarning: WMO_IDENTIFIER not set in the mission config (platform.WMO_IDENTIFIER) or the basestation file; writing 'UNK'.
  ds_OG1 = sg.convert(
/tmp/ipykernel_2522/3117069022.py:9: UserWarning: PLATFORM_DEPTH_RATING not set in the mission config (platform.PLATFORM_DEPTH_RATING); it is numeric, so the variable is omitted rather than written as 'UNK'.
  ds_OG1 = sg.convert(
Dimensions not in the OG1 dataset: ['trajectory']; their variables are not written (see conversion_dimensions).


,Attribute,Value,DType
0,conversion_units,"{""speed_gsm"": [""cm/s"", ""m s-1"", 0.01], ""horz_s...",str
1,conversion_sensor_calibration,"{""SENSOR_CTD_238007"": {""sensor_key"": ""legato"",...",str
2,title,OceanGliders trajectory file,str
3,id,sg271_20260302T193828_R,str
4,platform,sub-surface gliders,str
5,platform_vocabulary,https://vocab.nerc.ac.uk/collection/L06/curren...,str
6,geospatial_lat_min,-72.98122053363595,ndarray
7,geospatial_lat_max,-72.97946730813256,ndarray
8,geospatial_lon_min,171.4084666666667,ndarray
9,geospatial_lon_max,171.43743333333333,ndarray


### Convert mission from the NCEI server (with p*.nc files)

- Point `server` at an NCEI Seaglider mission directory and uncomment the cell below. A list of example missions is given further down.

In [12]:
# Specify the server where data are located
# server = "https://www.ncei.noaa.gov/data/oceans/glider/seaglider/uw/033/20100903/"

# Load and concatenate all datasets from the server, optionally specifying the range of profiles to load
# list_datasets = readers.read_mission(server, start_profile=1, end_profile=19)

# Convert the list of datasets to OG1
# ds_OG1 = sg.convert(list_datasets)

## Writing an output file

Due to problems with writing `xarray` datasets as netCDF when attributes are not of a specified type (`str`, `Number`, `np.ndarray`, `np.number`, `list`, `tuple`), a function was written `write`.

In [13]:
# Write the file
# This writer catches errors in data types (DType errors) when using xr.to_netcdf()
# The solution is to convert them to strings, which may be undesired behaviour
output_file = data_path / "demo_test.nc"
if output_file.exists():
    output_file.unlink()

writers.write(ds_OG1, output_file);

In [14]:
# Load the data saved
ds1 = xr.open_dataset(output_file)

# Generate a simple plot
# plotters.show_contents(ds_all,'attrs')
plotters.plot_depth_colored(ds1, color_by="PROFILE_NUMBER")

(<Figure size 1000x600 with 2 Axes>,
 <Axes: title={'center': 'Depth as a Function of Time'}, xlabel='Time (2026)', ylabel='Depth'>)

## One call per mission: `process` 

`sg.process` runs the whole pipeline for one mission from its configuration — read, convert, write `<output_dir>/<id>.nc` — and with `report=True` also writes the HTML conversion page beside it. It is what `seagliderOG1 process` calls for each mission.

Here it takes the bundled sg271 sample's configuration (`readers.sample_mission_dir()`), the same file `seagliderOG1 init --example` writes. Paths in it are resolved relative to the file; `output_dir` is overridden to `./outputs` below so the file is written in the repository.

In [15]:
# The bundled sg271 sample ships inside the package; process it, writing to ./outputs.
example = readers.sample_mission_dir() / "sg271_drygalski_2026.yaml"
path = sg.process(
    example, output_dir=parent_dir / "outputs", overwrite=True, report=True
)
print(path)

internal_mission_identifier: internal_mission_identifier is unset; it is OG1 highly desirable. Set it under global_attributes to improve discoverability.


site_vocabulary: site_vocabulary is unset; it is OG1 highly desirable. Set it under identity_and_discovery to improve discoverability.


program: program is unset; it is OG1 highly desirable. Set it under global_attributes to improve discoverability.


program_vocabulary: program_vocabulary is unset; it is OG1 highly desirable. Set it under global_attributes to improve discoverability.


data_url: data_url is unset; it is OG1 highly desirable. Set it under identity_and_discovery to improve discoverability.


doi: doi is unset; it is OG1 highly desirable. Set it under rights_and_access to improve discoverability.


... (6 outputs left out)

Processing datasets: 100%|██████████| 3/3 [00:01<00:00,  2.87dataset/s]

Processing datasets: 100%|██████████| 3/3 [00:01<00:00,  2.90dataset/s]

/home/runner/work/seagliderOG1/seagliderOG1/seagliderOG1/_convert.py:1563: UserWarning: identity_and_discovery.project overrides the converter value 'PycnMix_Drygalski' with 'PycnMix'.
  _apply_deployment(ds_og1, deployment, attribute_sources)
Dimensions not in the OG1 dataset: ['trajectory']; their variables are not written (see conversion_dimensions).


/home/runner/work/seagliderOG1/seagliderOG1/outputs/sg271_20260302T193828_R.nc
